In [ ]:
# import basic libraries

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

This document is used to display data visualization charts, calculate rolling statistics (rolling mean/SD) to check for drift, and compute basic statistical metrics for each factor ( mean, median, max value, min. value, and SD). Lastly, assess the presence of trends and seasonality.

The data in the ```final_monthly``` folder has been standardized for timestamps and cleaned; This notebook will double-check these cleanned data.

此文件用于展示数据可视化图表，计算滚动统计量（例如rolling mean/std）用于检查漂移，计算各个因子的基本统计数据（包括均值，中位数，最大值，最小值，标准差）。最后，判断是否包含趋势与季节性等走势。

```final_monthly```文件夹内的数据已经进行了时间戳统一和数据清洗，下面将检查数据是否可以直接使用。

In [ ]:
# dataset file path

DATASET_PATH = Path("data/final_monthly")
if not DATASET_PATH.is_dir():
    DATASET_PATH = Path("../data/final_monthly")
dataset_name_list = [
    "base_money_log_change_monthly.csv",
    "current_account_balance_annual_change_pp_monthly.csv",
    "food_imports_log_change_monthly.csv",
    "fuel_imports_log_change_monthly.csv",
    "fx_reserves_log_change_monthly.csv",
    "general_government_gross_debt_annual_change_pp_monthly.csv",
    "iron_ore_exports_log_change_monthly.csv",
    "NO_FILL_iron_ore_exports_log_change_monthly.csv",
    "USDSLE_monthly.csv"
]

默认使用2012—2024年训练期，每个CSV可视化图表包含原序列及12个月滚动均值、滚动标准差、去除线性斜率后的月份均值三个子图。

汇总表中，`screen_pass=True`表示本轮未检出设定范围内的信号，可继续ADF检验；`False`表示存在需要核查的信号或样本缺口；`<NA>`表示本方法不适用或数据不足。

存在中间缺失时，回归筛查仅使用最长连续区间，并在表中列明。其他月份没有被该回归检验覆盖。**年度变化逐月重复的两条序列仅展示描述性滚动图。**


By default, the 2012–2024 training period is used, and each CSV visualization chart contains three subplots: the original series with its 12-month rolling mean, the rolling standard deviation, and the monthly means after removing the linear slope.

In the summary table, `screen_pass=True` indicates that no signals within the configured scope were detected in this round, and the ADF test can proceed; `False` indicates that there are signals requiring investigation or sample gaps; `<NA>` indicates that the method is not applicable or the data are insufficient.

When there are internal gaps, the regression screening uses only the longest continuous interval and lists it in the table. The other months are not covered by this regression test. **For the two series whose annual changes repeat month by month, only descriptive rolling plots are shown.**

In [6]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from IPython.display import display

# Select settings before examining the held-out forecast period.
SCREEN_START = "2012-01-31"
SCREEN_END = "2024-12-31"
ROLLING_WINDOW = 12
MIN_MONTHS = 36
HAC_LAGS = 12
ALPHA = 0.05
# Exploratory effect-size thresholds, NOT statistical critical values.
MEAN_DRIFT_LIMIT = 1.0
STD_RATIO_LIMIT = 2.0


In [7]:
def prepare_monthly(file_path):
    # The first physical column is the date; only the next column is data.
    frame = pd.read_csv(file_path, usecols=[0, 1], index_col=0, parse_dates=[0])
    values = pd.to_numeric(frame.iloc[:, 0], errors="raise").sort_index()
    if values.index.hasnans or values.index.has_duplicates:
        raise ValueError(f"Invalid/duplicate dates: {file_path.name}")
    if not values.index.is_month_end.all():
        raise ValueError(f"Expected month-end dates: {file_path.name}")
    if np.isinf(values.to_numpy(dtype=float)).any():
        raise ValueError(f"Infinite values: {file_path.name}")
    calendar = pd.date_range(SCREEN_START, SCREEN_END, freq="ME")
    return values.reindex(calendar).rename(frame.columns[0])


def longest_complete_segment(values):
    # Never join observations across a missing calendar month.
    groups = values.isna().cumsum()
    segments = [part.dropna() for _, part in values.groupby(groups) if part.notna().any()]
    return max(segments, key=len) if segments else values.iloc[:0]


def screen_series(values, annual_repeated=False):
    rolling = pd.DataFrame({
        "mean": values.rolling(ROLLING_WINDOW, min_periods=ROLLING_WINDOW).mean(),
        "std": values.rolling(ROLLING_WINDOW, min_periods=ROLLING_WINDOW).std(ddof=1),
    })
    segment = longest_complete_segment(values)
    row = {
        "variable": values.name,
        "valid_months": int(values.notna().sum()),
        "missing_months": int(values.isna().sum()),
        "screen_start": segment.index.min() if len(segment) else pd.NaT,
        "screen_end": segment.index.max() if len(segment) else pd.NaT,
        "screen_n": len(segment),
        "trend_per_year": np.nan,
        "trend_p": np.nan,
        "seasonality_p": np.nan,
        "mean_drift_score": np.nan,
        "std_ratio": np.nan,
        "applicability": "monthly",
        "sample_note": "full training interval",
    }
    if annual_repeated:
        row["applicability"] = "annual values repeated monthly: monthly screen not applicable"
        return row, rolling, segment, None
    if len(segment) != len(values):
        row["sample_note"] = "longest complete interval only; other months not assessed by tests"
    if len(segment) < MIN_MONTHS:
        row["applicability"] = "insufficient consecutive months"
        return row, rolling, segment, None
    scale = segment.std(ddof=1)
    if not np.isfinite(scale) or scale == 0:
        row["applicability"] = "constant series: inference not applicable"
        return row, rolling, segment, None

    # Joint model: intercept + time in years + calendar-month indicators.
    # HAC allows short-run serial correlation/heteroskedasticity, not unit roots.
    design = pd.DataFrame({"const": 1.0, "time_years": np.arange(len(segment)) / 12.0}, index=segment.index)
    months = pd.Series(segment.index.month, index=segment.index)
    dummies = pd.get_dummies(months, prefix="month", drop_first=True, dtype=float)
    design = pd.concat([design, dummies], axis=1)
    fit = sm.OLS(segment, design).fit(
        cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS, "use_correction": True}
    )
    restrictions = np.zeros((len(dummies.columns), len(design.columns)))
    restrictions[:, 2:] = np.eye(len(dummies.columns))
    seasonal_test = fit.wald_test(restrictions, use_f=False, scalar=True)
    row["trend_per_year"] = float(fit.params["time_years"])
    row["trend_p"] = float(fit.pvalues["time_years"])
    row["seasonality_p"] = float(seasonal_test.pvalue)

    # Quantile ranges reduce sensitivity to a single extreme rolling window.
    # Both rolling scores refer to the SAME interval as the regression tests.
    segment_roll = rolling.loc[segment.index].dropna()
    q_mean = segment_roll["mean"].quantile([0.1, 0.9])
    q_std = segment_roll["std"].quantile([0.1, 0.9])
    row["mean_drift_score"] = float((q_mean.loc[0.9] - q_mean.loc[0.1]) / scale)
    low, high = float(q_std.loc[0.1]), float(q_std.loc[0.9])
    row["std_ratio"] = high / low if low > 0 else (np.inf if high > 0 else 1.0)
    trend = fit.params["const"] + fit.params["time_years"] * design["time_years"]
    # Remove just the slope when displaying the calendar-month pattern.
    detrended = segment - fit.params["time_years"] * design["time_years"]
    return row, rolling, segment, (trend, detrended)


def plot_screen(values, row, rolling, segment, fitted):
    fig, axes = plt.subplots(3, 1, figsize=(12, 9))
    values.plot(ax=axes[0], label="Observed", linewidth=0.9)
    rolling["mean"].plot(ax=axes[0], label=f"{ROLLING_WINDOW}-month rolling mean", linewidth=1.6)
    if fitted is not None:
        fitted[0].plot(ax=axes[0], label="Linear trend (base month)", linestyle="--")
    axes[0].set_title(f"{values.name} | {SCREEN_START[:7]} to {SCREEN_END[:7]}")
    axes[0].legend()
    rolling["std"].plot(ax=axes[1], label=f"{ROLLING_WINDOW}-month rolling SD", color="tab:orange")
    axes[1].legend()
    if fitted is not None:
        pattern = fitted[1].groupby(fitted[1].index.month).mean().reindex(range(1, 13))
        axes[2].plot(pattern.index, pattern.values, marker="o")
        axes[2].set_xticks(range(1, 13))
        axes[2].set_xlabel("Calendar month")
        axes[2].set_title("Mean by calendar month after removing the linear slope (screen interval)")
    else:
        axes[2].text(0.5, 0.5, row["applicability"], ha="center", va="center", wrap=True, transform=axes[2].transAxes)
        axes[2].set_axis_off()
    for ax in axes[:2]:
        ax.set_xlabel("Date")
    for ax in axes:
        ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
# One figure per CSV, with rolling and seasonal diagnostics in subplots.
plt.close("all")
plot_names = [
    name for name in dict.fromkeys(dataset_name_list)
    if name != "Sierra_Leone_Policy_Event_Dummies_monthly.csv"
]
screen_rows = []
rolling_statistics = {}
for dataset_name in plot_names:
    dataset_path = DATASET_PATH / dataset_name
    values = prepare_monthly(dataset_path)
    print(f"\n{dataset_path.stem} | training-period descriptive statistics")
    print(values.describe())
    annual_repeated = "_annual_change_pp_monthly" in dataset_path.stem
    row, rolling, segment, fitted = screen_series(values, annual_repeated)
    screen_rows.append(row)
    rolling_statistics[dataset_path.stem] = rolling
    plot_screen(values, row, rolling, segment, fitted)

print(f"Displayed {len(plot_names)} figures: one CSV per figure, three diagnostic panels.")


下面的汇总表用于检查数据中的线性趋势、月份季节效应，以及均值和波动的变化，帮助确定后续单位根检验的样本和设定。

- ```trend_per_year```描述存在的趋势，正值表示拟合均值随时间上升，负值表示下降。
- ```trend_p```用于检验线性趋势。其中p<0.05 表示单项检验下存在反对零斜率的证据；p≥0.05 表示未检出显著线性趋势
- ```seasonality_p```检验季节性效应。p<0.05 提示不同月份之间存在固定均值差异，可结合月份图检查；p≥0.05 表示未检出这类季节效应。
- ```mean_drift_score```滚动均值检查分数。当 ≥1.0 时提示核查均值漂移；<1.0 仅表示没有触发该阈值。
- ```std_ratio```滚动标准差分数。数与低分位数之比。	越接近1，所比较的滚动波动幅度越接近；当前 ≥2.0 时提示核查波动变化。
- ```screen_pass```综合检验结果、滚动指标、样本完整性和方法适用性得到的筛查标记。True：本轮未触发提示，可继续单位根检验；False：存在需核查的信号或样本缺口；<NA>：不适用或无法作出判断。


The summary table below is used to check for linear trends, monthly seasonality, and changes in mean and volatility in the data, to help determine the sample and specification for subsequent unit root tests.

- ```trend_per_year``` describes the trend present: a positive value indicates that the fitted mean rises over time, while a negative value indicates a decline.

- ```trend_p``` is used to test for a linear trend. A p-value < 0.05 indicates evidence against a zero slope under the individual test; a p-value ≥ 0.05 indicates that no significant linear trend was detected.

- ```seasonality_p``` tests for seasonality. A p-value < 0.05 suggests fixed mean differences across months, which can be checked together with the monthly plot; a p-value ≥ 0.05 indicates that no such seasonal effect was detected.

- ```mean_drift_score``` is a rolling mean check score. When ≥ 1.0, it flags the need to investigate mean drift; < 1.0 only means that this threshold was not triggered.

- ```std_ratio``` is a rolling standard deviation score: the ratio of the score to the lower quantile. The closer it is to 1, the closer the compared rolling volatility magnitudes are; when currently ≥ 2.0, it flags the need to investigate volatility changes.

- ```screen_pass``` is a screening flag derived from the combined test results, rolling indicators, sample completeness, and method applicability. True: no warning was triggered in this round, and unit root testing can proceed; False: there are signals requiring investigation or sample gaps; <NA>: not applicable or unable to make a judgment.

In [ ]:
screening_results = pd.DataFrame(screen_rows)
# Holm correction across all trend + seasonal tests in this screen.
p_columns = ["trend_p", "seasonality_p"]
p_values = screening_results[p_columns].to_numpy(dtype=float)
finite = np.isfinite(p_values)
adjusted = np.full_like(p_values, np.nan)
if finite.any():
    adjusted[finite] = multipletests(p_values[finite], alpha=ALPHA, method="holm")[1]
screening_results[["trend_p_holm", "seasonality_p_holm"]] = adjusted


def interpret_screen(row):
    if row["applicability"] != "monthly":
        return pd.Series({"screen_pass": pd.NA, "screen_status": "not assessed", "next_step": row["applicability"]})
    if not np.isfinite(row["trend_p_holm"]) or not np.isfinite(row["seasonality_p_holm"]):
        return pd.Series({"screen_pass": pd.NA, "screen_status": "not assessed", "next_step": "inspect undefined test statistics"})
    signals = []
    if row["trend_p_holm"] < ALPHA:
        signals.append("linear trend")
    if row["seasonality_p_holm"] < ALPHA:
        signals.append("calendar-month seasonality")
    if row["mean_drift_score"] >= MEAN_DRIFT_LIMIT:
        signals.append("rolling mean drift")
    if row["std_ratio"] >= STD_RATIO_LIMIT:
        signals.append("rolling SD variation")
    if row["missing_months"]:
        signals.append("incomplete training interval")
    passed = not signals
    status = "no signal detected under this screen" if passed else "; ".join(signals)
    next_step = "proceed to ADF/KPSS; stationarity is NOT established" if passed else "review signals and choose ADF/KPSS sample and deterministic terms"
    return pd.Series({"screen_pass": passed, "screen_status": status, "next_step": next_step})

screening_results = pd.concat([screening_results, screening_results.apply(interpret_screen, axis=1)], axis=1)
screening_results["screen_pass"] = screening_results["screen_pass"].astype("boolean")
# Hide these columns in the displayed table; retain them for calculations.
hidden_columns = [
    "missing_months", "screen_start", "screen_end", "screen_n",
    "applicability", "sample_note", "trend_p_holm", "seasonality_p_holm",
    "screen_status", "next_step",
]
with pd.option_context("display.max_columns", None, "display.max_colwidth", 90):
    display(screening_results.drop(columns=hidden_columns))
